In [ ]:
# Add project root to Python path
# This is needed so the notebook can import from src/ folder
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')
print('Path configured successfully.')

In [ ]:
from src.data_loader   import load_dataset
from src.preprocessing import run_preprocessing
from src.eda           import run_eda
from src.models        import train_decision_tree, train_random_forest
from src.evaluation    import (
    evaluate_model, plot_confusion_and_roc,
    plot_decision_tree_visual, plot_feature_importance,
    plot_comparison, run_cross_validation
)
from src.predictor     import predict_rain_for_date, plot_prediction_chart

import warnings
warnings.filterwarnings('ignore')

print('All modules imported successfully.')

In [ ]:
# Load the dataset and display the first five rows
df_raw = load_dataset()
df_raw.head()

In [ ]:
# Run EDA (before preprocessing)
# We run EDA on raw data to understand it before any transformation
#
run_eda(df_raw)

In [ ]:
# Run preprocessing to identify available features, missing values and split data to train set and test set

df, X_train_scaled, X_test_scaled, y_train, y_test, scaler, feature_cols = \
    run_preprocessing(df_raw.copy())

In [ ]:
# Run preprocessing to identify available features, missing values and split data to train set and test set
df, X_train_scaled, X_test_scaled, y_train, y_test, scaler,
feature_cols = \
run_preprocessing(df_raw.copy())

In [ ]:
# EDA after preprocessing to display correlation heatmap with final features
import pandas as pd
X_full = df[feature_cols].fillna(df[feature_cols].median())
from src.eda import plot_correlation_heatmap
plot_correlation_heatmap(X_full)

In [ ]:
# Algorithm 1
# Train Decision Tree
dt_model = train_decision_tree(X_train_scaled, y_train)

In [ ]:
# Evaluate Decision Tree
# evaluate_model() will returns all metrics needed for comparison
dt_metrics = evaluate_model(dt_model, X_test_scaled, y_test, 'Decision Tree')

In [ ]:
# visualize Confusion Matrix and ROC Curve
plot_confusion_and_roc(dt_metrics, cmap='Blues', y_test=y_test)

In [ ]:
# Visualize the tree (top 3 levels)
plot_decision_tree_visual(dt_model, feature_cols)

In [ ]:
# Algorithm 2
# Train Random Forest
rf_model = train_random_forest(X_train_scaled, y_train)

In [ ]:
# Evaluate Random Forest
rf_metrics = evaluate_model(rf_model, X_test_scaled, y_test, 'Random Forest')

In [ ]:
# Confusion Matrix + ROC Curve
plot_confusion_and_roc(rf_metrics, cmap='Greens', y_test=y_test)

In [ ]:
# Feature Importances
plot_feature_importance(rf_model, feature_cols)

In [ ]:
##  Algorithm Comparison
# Compare both algorithms
plot_comparison(dt_metrics, rf_metrics)

In [ ]:
# 5-Fold Cross Validation
run_cross_validation(dt_model, rf_model, X_train_scaled, y_train)

In [ ]:
# PREDICT FOR ANY DATE AND CITY

QUERY_DATE = '2021-06-15'    # Format: YYYY-MM-DD
QUERY_CITY = 'Colombo'       # city name


predict_rain_for_date(
    input_date   = QUERY_DATE,
    input_city   = QUERY_CITY,
    df           = df,
    dt_model     = dt_model,
    rf_model     = rf_model,
    scaler       = scaler,
    feature_cols = feature_cols
)

In [ ]:
#  Visualise prediction as probability bar chart
plot_prediction_chart(
    input_date   = QUERY_DATE,
    input_city   = QUERY_CITY,
    df           = df,
    dt_model     = dt_model,
    rf_model     = rf_model,
    scaler       = scaler,
    feature_cols = feature_cols
)

In [ ]:
#  Predict for multiple dates and cities
test_cases = [
    ('2020-10-20', 'Colombo'),
    ('2018-12-01', 'Kandy'),
    ('2015-04-10', 'Galle'),
    ('2019-08-25', 'Jaffna'),
]

for date, city in test_cases:
    predict_rain_for_date(date, city, df, dt_model, rf_model, scaler, feature_cols)
    print()